# Calibration loss by region, batch and epoch: harvest date and yield

The objective is a **sum of per-region means**: one forward pass, then each
region's loss on its own rows. So a mixed-region batch gives every region
exactly the gradient its own batch would. It also means the per-region terms
plotted here are the quantities the optimiser actually sees.
`<stage>/batch_history.parquet` keeps them, one row per epoch, split, batch,
region and term. `history.csv` averages them away.

| Term | Meaning |
|---|---|
| `phenology/harvest_date_rmse_d` | Unit-mean simulated maturity (DVS 2 crossing) against the observed harvest date [d] |
| `phenology/harvest_date_bias_d` | Simulated minus observed [d]; positive is late |
| `yield/yield_rmse_t_ha` | Unit-mean dry-matter yield against the detrended `yield_level` [t/ha] |
| `loss` | The region's weighted objective, all active terms, retention included |

**SIMPLACE is not calibrated.** It re-runs on the crop files torchcrop fits, so
it appears as horizontal references: once on the default crop and once on the
per-region calibrated crops. Both are scored by
`evaluation.calibration_loss.score_simplace` on the **same pool, observations
and validation split** as torchcrop. Maturity stands in for harvest in both
models.

## 0. Setup

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cropmodelling4eu.evaluation.style import (
    DIVERGING_CMAP, SEQUENTIAL_CMAP, diverging_norm, use_style,
)

logging.basicConfig(level=logging.WARNING, force=True)
P = use_style("light")
RUN = Path("/data01/FDS/muduchuru/Data/SIMPLACE/cropmodelling4eu/winter_wheat_2000_2024_euval")

from cropmodelling4eu.evaluation.calibration_loss import (
    load_batch_history, load_simplace_runs, score_simplace,
)
from cropmodelling4eu.evaluation.plots import cell_map

#: Calibration runs to compare, e.g. the CPU fork-pool run and the GPU mixed run.
RUNS = {
    "cpu mixed": RUN / "calibration_mixcheck_cpu",
}
STAGE = "phenology"
TERMS = {
    "phenology/harvest_date_rmse_d": "harvest-date RMSE [d]",
    "yield/yield_rmse_t_ha": "yield RMSE [t/ha]",
}
#: SIMPLACE on the default crop, and on the per-region calibrated crops.
SIMPLACE = {
    # `simplace/`, not the top-level RUN/simplace_europe.parquet: that one is
    # the region-merged *calibrated* run, so it would score calibrated twice.
    "SIMPLACE default": [RUN / "simplace" / "simplace_europe.parquet"],
    "SIMPLACE calibrated": sorted((RUN / "simplace_regions").glob("region_*/simplace_europe.parquet")),
}

history = pd.concat(
    [load_batch_history(path, [STAGE]).assign(run=name) for name, path in RUNS.items()],
    ignore_index=True,
)
TERMS = {k: v for k, v in TERMS.items() if k in set(history["term"])}
primary = next(iter(RUNS))
regions = pd.read_parquet(RUNS[primary] / "regions.parquet")
history.groupby(["run", "split"])[["epoch", "batch"]].max()

## 1. SIMPLACE references on the same split

This cell is skipped with a note if the run predates `pool.parquet` and
`<stage>_split.parquet`. Those files are written by every calibration since
mixed-region batching.

In [ ]:
reference = pd.DataFrame()
needed = [RUNS[primary] / n for n in ("pool.parquet", "observations.parquet", f"{STAGE}_split.parquet")]
if all(p.is_file() for p in needed):
    reference = pd.concat(
        [score_simplace(RUNS[primary], load_simplace_runs(paths), STAGE, label)
         for label, paths in SIMPLACE.items() if paths],
        ignore_index=True,
    )
else:
    print("no persisted pool/split under", RUNS[primary], "- SIMPLACE references skipped")

# A calibrated run that reproduces the default one exactly did not apply its
# crops; say so here rather than draw one line on top of the other.
if reference["label"].nunique() == 2:
    pair = reference.pivot_table(index=["split", "region_id", "term"], columns="label", values="value")
    same = (pair.iloc[:, 0] - pair.iloc[:, 1]).abs().lt(1e-9).mean()
    if same > 0.99:
        print(f"WARNING: SIMPLACE default and calibrated agree on {same:.0%} of terms - "
              "check the per-region runs took their crops, and that 'default' is RUN/simplace/")
reference.head()

## 2. Region × epoch

Each cell is the row-weighted mean over that epoch's training batches for one
region. Magnitude uses a sequential ramp; the colour scale is shared within each
term, not across terms.

In [ ]:
train = history[(history["split"] == "train") & (history["run"] == primary)]
rows = train[train["term"] == "rows"][["epoch", "batch", "region_id", "value"]].rename(
    columns={"value": "w"})
for term, label in TERMS.items():
    block = train[train["term"] == term].merge(rows, on=["epoch", "batch", "region_id"])
    grid = (block.assign(wv=block["value"] * block["w"])
            .groupby(["region_id", "epoch"])[["wv", "w"]].sum()
            .pipe(lambda g: g["wv"] / g["w"]).unstack("epoch"))
    fig, ax = plt.subplots(figsize=(1.2 + 0.45 * grid.shape[1], 0.6 + 0.22 * grid.shape[0]))
    image = ax.imshow(grid.to_numpy(), aspect="auto", cmap=SEQUENTIAL_CMAP)
    ax.set(xticks=range(grid.shape[1]), xticklabels=grid.columns,
           yticks=range(grid.shape[0]), yticklabels=grid.index,
           xlabel="epoch", ylabel="region", title=f"{label}, training batches")
    fig.colorbar(image, ax=ax, label=label, shrink=0.8)
    fig.tight_layout()

## 3. Per region: every batch, the epoch mean, validation, SIMPLACE

The dots are individual training batches, each placed at its epoch plus its
position within that epoch. The line is the epoch mean and the diamonds are
validation. Horizontal lines are SIMPLACE on the validation split, solid for the
calibrated crops and dotted for the default crop.

In [ ]:
for term, label in TERMS.items():
    block = history[(history["term"] == term) & (history["run"] == primary)]
    ids = sorted(block["region_id"].unique())
    ncol = 6
    fig, axes = plt.subplots(-(-len(ids) // ncol), ncol, figsize=(13, 1.9 * -(-len(ids) // ncol)),
                             sharex=True, squeeze=False)
    for ax, region in zip(axes.flat, ids):
        own = block[block["region_id"] == region]
        tr = own[own["split"] == "train"].copy()
        span = tr.groupby("epoch")["batch"].transform("max").clip(lower=1)
        ax.scatter(tr["epoch"] + 0.8 * tr["batch"] / span, tr["value"], s=8,
                   color=P["muted"], linewidths=0)
        ax.plot(*zip(*tr.groupby("epoch")["value"].mean().items()), color=P["series_1"], lw=2)
        va = own[own["split"] == "val"].groupby("epoch")["value"].mean()
        ax.plot(va.index, va.values, marker="D", ms=4, lw=0, color=P["series_2"])
        if not reference.empty:
            ref = reference[(reference["split"] == "val") & (reference["term"] == term)
                            & (reference["region_id"] == region)]
            for _, r in ref.iterrows():
                ax.axhline(r["value"], color=P["series_3"], lw=1.5,
                           ls="-" if "calibrated" in r["label"] else ":")
        ax.set_title(f"region {region}", fontsize=8)
    for ax in axes.flat[len(ids):]:
        ax.set_visible(False)
    handles = [plt.Line2D([], [], color=P["muted"], marker="o", lw=0, label="train batch"),
               plt.Line2D([], [], color=P["series_1"], lw=2, label="train epoch mean"),
               plt.Line2D([], [], color=P["series_2"], marker="D", lw=0, label="validation"),
               plt.Line2D([], [], color=P["series_3"], lw=1.5, label="SIMPLACE calibrated"),
               plt.Line2D([], [], color=P["series_3"], lw=1.5, ls=":", label="SIMPLACE default")]
    fig.legend(handles=handles, loc="upper center", ncol=5, frameon=False, fontsize=8)
    fig.supylabel(label)
    fig.tight_layout(rect=(0, 0, 1, 0.96))

## 4. Where the loss sits on the map

These are the last validation epoch's per-region terms, painted onto every cell
of the region: torchcrop, then SIMPLACE on the calibrated crops, then the change
in torchcrop since epoch 0. A sequential ramp is used for levels and a diverging
one centred on zero for the change, where blue is improvement.

In [ ]:
val = history[(history["split"] == "val") & (history["run"] == primary)]
cells = regions[["SimplaceID", "lon", "lat", "region_id"]]
for term, label in TERMS.items():
    by_epoch = val[val["term"] == term].groupby(["epoch", "region_id"])["value"].mean().unstack("epoch")
    if by_epoch.empty:
        continue
    last, first = by_epoch.iloc[:, -1], by_epoch.iloc[:, 0]
    frame = cells.assign(
        torchcrop=cells["region_id"].map(last),
        change=cells["region_id"].map(last - first),
    )
    if not reference.empty:
        cal = reference[(reference["label"] == "SIMPLACE calibrated") & (reference["split"] == "val")
                        & (reference["term"] == term)].set_index("region_id")["value"]
        frame["simplace"] = cells["region_id"].map(cal)
    vmax = np.nanpercentile(frame[[c for c in ("torchcrop", "simplace") if c in frame]], 98)
    cell_map(frame.dropna(subset=["torchcrop"]), "torchcrop", cmap=SEQUENTIAL_CMAP, vmin=0,
             vmax=vmax, title=f"torchcrop, last epoch", cbar_label=label)
    if "simplace" in frame:
        cell_map(frame.dropna(subset=["simplace"]), "simplace", cmap=SEQUENTIAL_CMAP, vmin=0,
                 vmax=vmax, title="SIMPLACE, calibrated crops", cbar_label=label)
    cell_map(frame.dropna(subset=["change"]), "change", cmap=DIVERGING_CMAP,
             norm=diverging_norm(frame["change"].dropna()),
             title="torchcrop, change since epoch 0", cbar_label=f"Δ {label}")

## 5. Loss against wall-clock time

This compares the same stage run two ways, for example the CPU fork pool and
the GPU with mixed batches. Each point is one epoch: x is cumulative training
time, from the per-batch `seconds`, and y is the row-weighted validation term.

In [ ]:
for term, label in TERMS.items():
    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    for i, (run, block) in enumerate(history.groupby("run")):
        tr = block[(block["split"] == "train") & (block["term"] == term)]
        seconds = tr.drop_duplicates(["epoch", "batch"]).groupby("epoch")["seconds"].sum().cumsum()
        va = block[(block["split"] == "val") & (block["term"] == term)].groupby("epoch")["value"].mean()
        joined = pd.concat([seconds.rename("s"), va.rename("v")], axis=1).dropna()
        ax.plot(joined["s"] / 60, joined["v"], marker="o", ms=4, lw=2,
                color=P[f"series_{1 + i}"], label=run)
    ax.set(xlabel="training wall time [min]", ylabel=f"validation {label}")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()

## 6. Table view

In [ ]:
summary = (history[history["term"].isin(TERMS)]
           .groupby(["run", "split", "term", "region_id", "epoch"])["value"].mean()
           .unstack("epoch"))
pd.concat([summary, reference.pivot_table(index=["label", "split", "term", "region_id"],
                                          values="value")
           if not reference.empty else None]).round(2)